# Lesson 22 — 配置落盘与版本迁移

配置必须跨进程重启保留，并能从旧格式迁移。`%config_store` 会真实写入项目沙箱；v1 → v2 的迁移由 C++ `ConfigStore` 完成。ESP32 对应 NVS 事务，这个 host 文件不是 Flash 模拟。

**学习顺序：** 先运行 guided，再修改，故意运行 broken，最后自己完成 exercise。答案 Notebook 独立保存。

In [ ]:
from pathlib import Path
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools/notebook_setup.py").is_file())
exec((project / "tools/notebook_setup.py").read_text())

## 1. Guided

先观察面板输出，再读 C++。

In [ ]:
%%academy_lab 22-config-migration guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() {
            const int interval = device_.input.analog * 1000;
            device_.output.error =
                interval < 1000 || interval > 3600000 || device_.input.text.empty();
            device_.output.state =
                device_.output.error ? "invalid-config" : "ready-to-save";
            device_.output.display =
                device_.input.text + " / " + std::to_string(interval) + " ms";
        }

    private:
        Device& device_;
    };
}

## 2. Modify

把输入文本改为 `desk-notifier`，再用 `%config_store` 点击写入配置。关闭并重新运行时观察文件仍在。

In [ ]:
%%academy_lab 22-config-migration modify
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() {
            const int interval = device_.input.analog * 1000;
            device_.output.error =
                interval < 1000 || interval > 3600000 || device_.input.text.empty();
            device_.output.state =
                device_.output.error ? "invalid-config" : "ready-to-save";
            device_.output.display =
                device_.input.text + " / " + std::to_string(interval) + " ms";
        }

    private:
        Device& device_;
    };
}

## 3. Broken

下面的检查应该失败；先观察故障证据，再查看分层提示。

In [ ]:
%%academy_lab 22-config-migration broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() {
            device_.output.error = device_.input.text.empty();
            device_.output.state = "ready-to-save";
        }

    private:
        Device& device_;
    };
}

## 4. Build

要求：拒绝空设备名和范围外间隔；用真实 `ConfigStore` 保存 v2；读取 v1 时把秒迁移为毫秒并重新落盘。

提示 1：先 validate。提示 2：v1 的 `interval_s * 1000`。提示 3：临时文件写完后再原子替换。

In [ ]:
%%academy_lab 22-config-migration exercise exercise
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() { /* TODO: 你的实现 */
        }

    private:
        Device& device_;
    };
}


## 5. 真实配置沙箱

In [ ]:
%config_store

## 6. 反思

写下：这个模型在 host 上证明了什么？哪些行为必须买板后重新验证？